In [9]:
import os
import pandas as pd
import numpy as np


In [10]:

def convert_acceleration(voltage):
    """Convert accelerometer voltage to gravities."""
    return round((voltage - 1.65) * (5.0 / 3.3), 6)


In [11]:

def convert_gyroscope(voltage, zero_value):
    """Convert gyroscope voltage to degrees per second."""
    return round((voltage - zero_value) * 400, 6)


In [12]:

def process_scale_values(scale_data):
    """Replace 0.0 values in the scale data with the average of surrounding non-zero values."""
    non_zero_values = scale_data[scale_data != 0]
    average_value = non_zero_values.mean()
    scale_data[scale_data == 0] = average_value
    return scale_data


In [13]:
def read_all_data_files(data_folder):
    data_records = []

    # Loop through each user directory
    for user_id in os.listdir(data_folder):
        user_path = os.path.join(data_folder, user_id)
        
        if os.path.isdir(user_path):
            # Loop through each course directory
            for course in os.listdir(user_path):
                course_path = os.path.join(user_path, course)
                
                if os.path.isdir(course_path):
                    # Loop through each data file in the course directory
                    for data_file in os.listdir(course_path):
                        if data_file.endswith('.txt'):
                            data_path = os.path.join(course_path, data_file)
                            data_df = pd.read_csv(data_path, header=None, delimiter='\t')

                            # Convert accelerometer data (columns 1-3) to gravities
                            data_df[[0, 1, 2]] = data_df[[0, 1, 2]].map(convert_acceleration)

                            # Calculate zero value for gyroscope data (columns 4-6)
                            zero_values = data_df[[3, 4, 5]].mean()
                            # Convert gyroscope data to degrees per second
                            data_df[[3, 4, 5]] = data_df[[3, 4, 5]].apply(lambda col: convert_gyroscope(col, zero_values[col.name]))

                            # Add user_id, course, and index columns
                            data_df['user_id'] = user_id
                            data_df['course'] = course
                            data_df['index'] = data_df.index

                            # Rename columns for clarity
                            data_df.columns = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z', 'grams', 'user_id', 'course', 'index']

                            data_records.append(data_df)

    # Concatenate all data into a single DataFrame
    all_data_df = pd.concat(data_records, ignore_index=True)
    return all_data_df

In [14]:
def add_activity_labels(all_data_df, gt_folder):
    # Loop through each gesture file to match and label activities
    for user_id in os.listdir(gt_folder):
        user_path = os.path.join(gt_folder, user_id)
        
        if os.path.isdir(user_path):
            # Loop through each course directory
            for course in os.listdir(user_path):
                course_path = os.path.join(user_path, course)
                
                if os.path.isdir(course_path):
                    # Loop through each gesture file in the course directory
                    for gt_file in os.listdir(course_path):
                        if gt_file.endswith('.txt'):
                            gt_path = os.path.join(course_path, gt_file)
                            gt_df = pd.read_csv(gt_path, header=None, delimiter='\t')

                            # Filter the main DataFrame to match the current user and course
                            mask = (all_data_df['user_id'] == user_id) & (all_data_df['course'] == course)

                            # Initialize the activity column with default value 'other'
                            all_data_df.loc[mask, 'activity'] = 'other'

                            # Assign activities based on gesture ground truth
                            for _, row in gt_df.iterrows():
                                gesture_type = row[0]
                                start_idx = int(row[1])
                                end_idx = int(row[2])

                                # Update activity labels in the main DataFrame based on the index range
                                all_data_df.loc[mask & all_data_df['index'].between(start_idx, end_idx), 'activity'] = gesture_type

    return all_data_df

In [15]:
def combine_clemson_cafeteria_data(data_folder, gt_folder, output_file):
    # Step 1: Read all data files and create the initial DataFrame
    all_data_df = read_all_data_files(data_folder)

    print(all_data_df.head())
    
    # Step 2: Add activity labels to the DataFrame based on gesture files
    labeled_data_df = add_activity_labels(all_data_df, gt_folder)

    print(labeled_data_df.head())
    
    return labeled_data_df

In [16]:
# Example usage
data_folder = "C:\\Users\\tulio\\Downloads\\all-data"
gt_folder = "C:\\Users\\tulio\\Downloads\\all-gt-gestures"
output_file = 'combined_clemson_cafeteria_data.csv'

final_df = combine_clemson_cafeteria_data(data_folder, gt_folder, output_file)

      acc_x     acc_y     acc_z      gyr_x      gyr_y      gyr_z  grams  \
0 -0.074242 -0.127273  1.125758  65.232882  23.322505  46.738401  828.7   
1  0.392424 -0.016667  1.181818  49.632882  -5.077495  23.938401  828.7   
2  0.201515 -0.009091  1.127273  38.432882  -7.477495  17.938401  828.7   
3  0.251515 -0.046970  1.127273  38.832882  -9.877495  18.738401  828.7   
4  0.312121 -0.028788  1.122727  37.632882 -11.477495  19.138401  828.9   

  user_id course  index  
0    p005     c1      0  
1    p005     c1      1  
2    p005     c1      2  
3    p005     c1      3  
4    p005     c1      4  
      acc_x     acc_y     acc_z      gyr_x      gyr_y      gyr_z  grams  \
0 -0.074242 -0.127273  1.125758  65.232882  23.322505  46.738401  828.7   
1  0.392424 -0.016667  1.181818  49.632882  -5.077495  23.938401  828.7   
2  0.201515 -0.009091  1.127273  38.432882  -7.477495  17.938401  828.7   
3  0.251515 -0.046970  1.127273  38.832882  -9.877495  18.738401  828.7   
4  0.312121 -0.028

In [17]:
final_df

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,grams,user_id,course,index,activity
0,-0.074242,-0.127273,1.125758,65.232882,23.322505,46.738401,828.7,p005,c1,0,other
1,0.392424,-0.016667,1.181818,49.632882,-5.077495,23.938401,828.7,p005,c1,1,other
2,0.201515,-0.009091,1.127273,38.432882,-7.477495,17.938401,828.7,p005,c1,2,other
3,0.251515,-0.046970,1.127273,38.832882,-9.877495,18.738401,828.7,p005,c1,3,other
4,0.312121,-0.028788,1.122727,37.632882,-11.477495,19.138401,828.9,p005,c1,4,other
...,...,...,...,...,...,...,...,...,...,...,...
6367686,1.037879,-0.042424,0.013636,-5.460823,-30.588564,4.409891,826.2,p413,c1,17222,other
6367687,1.127273,-0.042424,-0.013636,-5.060823,-19.388564,4.809891,826.3,p413,c1,17223,other
6367688,1.078788,-0.007576,-0.015152,-12.660823,-12.988564,0.809891,826.3,p413,c1,17224,other
6367689,1.096970,0.019697,-0.066667,-10.260823,-14.188564,1.209891,826.3,p413,c1,17225,other


In [18]:
  # Save the final DataFrame to a CSV file
final_df.to_csv(output_file, index=False)
print(f"Combined data saved to {output_file}")

Combined data saved to combined_clemson_cafeteria_data.csv
